In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from LegendreKANLayer import LegendreKANLayer
import random

# 设置设备为 GPU，如果可用
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 设置随机种子
seed = 5
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

# 定义 Monge-Ampère 方程的源项和解析解
def source_function(x, y):
    return (1 + x**2 + y**2) * torch.exp(x**2 + y**2)

def analytical_solution(x, y):
    return torch.exp((x**2 + y**2) / 2)

# 定义求解模型
class LegenKAN(nn.Module):
    def __init__(self):
        super(LegenKAN, self).__init__()
        self.legenkan1 = LegendreKANLayer(2, 8, 6)
        self.legenkan2 = LegendreKANLayer(8, 8, 6)
        self.legenkan3 = LegendreKANLayer(8, 1, 6)

    def forward(self, x, y):
        xy = torch.cat([x, y], dim=1)
        xy = self.legenkan1(xy)
        xy = self.legenkan2(xy)
        xy = self.legenkan3(xy)
        return xy

# 将模型放到 GPU
solver = LegenKAN().to(device)

# 初始采样点
x_values = torch.linspace(0, 1, 200).view(-1, 1).to(device)
y_values = torch.linspace(0, 1, 200).view(-1, 1).to(device)
X, Y = torch.meshgrid(x_values.squeeze(), y_values.squeeze(), indexing="ij")
X, Y = X.reshape(-1, 1), Y.reshape(-1, 1)

# 内部点和边界点的掩码
boundary_mask = (X == 0) | (X == 1) | (Y == 0) | (Y == 1)
interior_mask = ~boundary_mask

X_boundary, Y_boundary = X[boundary_mask].unsqueeze(1), Y[boundary_mask].unsqueeze(1)
X_interior, Y_interior = X[interior_mask].unsqueeze(1), Y[interior_mask].unsqueeze(1)

# 启用梯度
X_interior.requires_grad = True
Y_interior.requires_grad = True

# 损失函数和优化器
criterion = nn.MSELoss()
optimizer = optim.Adam(solver.parameters(), lr=0.01)

# 自适应采样参数
num_adaptive_steps = 5
num_high_error_samples = 200
learning_rate_decay = 0.8
adaptive_sample_increment = 20

loss_LK = []

# 训练
epochs = 10000
alpha = 0.00001
previous_loss = float('inf')

for epoch in range(epochs):
    optimizer.zero_grad()

    # 计算边界和内部点的数值解
    numerical_boundary = solver(X_boundary, Y_boundary)
    boundary_target = analytical_solution(X_boundary, Y_boundary)  # 边界上的解析解

    # 边界损失，确保满足边界条件
    boundary_loss = criterion(numerical_boundary, boundary_target)
    numerical_interior = solver(X_interior, Y_interior)

    # 内部点的 Monge-Ampère 方程损失
    u_x = torch.autograd.grad(numerical_interior, X_interior, grad_outputs=torch.ones_like(numerical_interior), create_graph=True)[0]
    u_y = torch.autograd.grad(numerical_interior, Y_interior, grad_outputs=torch.ones_like(numerical_interior), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, X_interior, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, Y_interior, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    u_xy = torch.autograd.grad(u_x, Y_interior, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]

    hessian_det = u_xx * u_yy - u_xy**2
    interior_loss = criterion(hessian_det, source_function(X_interior, Y_interior))

    # 总损失
    loss = boundary_loss + alpha * interior_loss

    # 反向传播与优化
    loss.backward(retain_graph=True)
    optimizer.step()

    # 添加当前的损失值到列表中
    loss_LK.append(loss.item())

    # 检查损失是否增加，增加时减少学习率
    if loss.item() > previous_loss:
        for param_group in optimizer.param_groups:
            param_group['lr'] *= learning_rate_decay
        print(f"Epoch [{epoch+1}/{epochs}], Loss increased. Reducing learning rate to: {param_group['lr']:.6f}")

    # 更新前一轮的总损失
    previous_loss = loss.item()

    # 每200个epoch输出损失信息
    if (epoch + 1) % 200 == 0:
        numerical_solution_global = solver(X, Y)
        analytical_solution_global = analytical_solution(X, Y)
        error_global = torch.abs(numerical_solution_global - analytical_solution_global)

        max_error = torch.max(error_global).item()
        mean_error = torch.mean(error_global).item()
        l2_error = torch.sqrt(torch.mean((numerical_solution_global - analytical_solution_global) ** 2)).item()

        print(f"Epoch [{epoch+1}/{epochs}], Total Loss: {loss.item():.4e}, "
            f"Boundary Loss: {boundary_loss.item():.4e}, Interior Loss: {interior_loss.item():.4e}, "
            f"Max Error: {max_error:.4e}, Average Error: {mean_error:.4e}, "
            f"L2 Error: {l2_error:.4e}")
        
    # 自适应采样
    if (epoch + 1) % (epochs // num_adaptive_steps) == 0:
        # 使用当前模型预测
        numerical_solution = solver(X_interior, Y_interior)

        # 计算一阶导数
        u_x = torch.autograd.grad(numerical_solution, X_interior, grad_outputs=torch.ones_like(numerical_solution), create_graph=True)[0]
        u_y = torch.autograd.grad(numerical_solution, Y_interior, grad_outputs=torch.ones_like(numerical_solution), create_graph=True)[0]

        # 计算二阶导数
        u_xx = torch.autograd.grad(u_x, X_interior, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
        u_yy = torch.autograd.grad(u_y, Y_interior, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
        u_xy = torch.autograd.grad(u_x, Y_interior, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]

        # 计算残差（预测的 Hessian 行列式 - 源项）
        hessian_det = u_xx * u_yy - u_xy**2
        residual = torch.abs(hessian_det - source_function(X_interior, Y_interior))

        # 选取残差最大的点
        _, high_error_indices = torch.topk(residual.view(-1), num_high_error_samples)
        X_high_error, Y_high_error = X_interior[high_error_indices], Y_interior[high_error_indices]

        # 以这些点为中心采样新点
        delta = 0.05
        X_new = X_high_error + (torch.rand_like(X_high_error) - 0.5) * delta
        Y_new = Y_high_error + (torch.rand_like(Y_high_error) - 0.5) * delta

        # 限制在 [0, 1] 范围内，防止越界
        X_new = torch.clamp(X_new, 0.0, 1.0)
        Y_new = torch.clamp(Y_new, 0.0, 1.0)

        # 添加新点
        X_interior = torch.cat([X_interior, X_new.requires_grad_(True)])
        Y_interior = torch.cat([Y_interior, Y_new.requires_grad_(True)])
        num_high_error_samples += adaptive_sample_increment


# 生成损失图
plt.figure(figsize=(10, 5))
plt.plot(range(1, epochs + 1), loss_LK)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training Loss over Epochs')
plt.show()

# 绘图
numerical_solution = solver(X, Y)
analytical_solution_values = analytical_solution(X, Y)
error = torch.abs(numerical_solution - analytical_solution_values)
max_error = torch.max(error).item()
print(f"Max Error: {max_error:.4e}")
plt.figure(figsize=(18, 5))

# 数值解图
plt.subplot(1, 3, 1)
plt.tricontourf(X.view(-1).detach().cpu().numpy(), Y.view(-1).detach().cpu().numpy(),
                numerical_solution.detach().cpu().numpy().squeeze(), levels=20, cmap="viridis")
plt.colorbar()
plt.title("Numerical Solution")

# 解析解图
plt.subplot(1, 3, 2)
plt.tricontourf(X.view(-1).detach().cpu().numpy(), Y.view(-1).detach().cpu().squeeze().numpy(),
                analytical_solution_values.detach().cpu().numpy().squeeze(), levels=20, cmap="viridis")
plt.colorbar()
plt.title("Analytical Solution")

# 误差图
plt.subplot(1, 3, 3)
plt.tricontourf(X.view(-1).detach().cpu().numpy(), Y.view(-1).detach().cpu().numpy(),
                error.detach().cpu().numpy().squeeze(), levels=40, cmap="inferno")
plt.colorbar()
plt.title("Error between Numerical and Analytical Solution")

plt.show()